# 🤖 Jarvis_liquid_agent — free cloud Jarvis (Kaggle / Colab)
Runs **LFM3-1.7B-Uncensored** (GGUF, Q4_K_M) on a free T4 GPU with official llama.cpp binaries, and gives you a public **trycloudflare.com** link you can open on your phone (no accounts needed).

**Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100), **Internet ON** (needs a phone-verified Kaggle account).  
**Colab:** Runtime → Change runtime type → **T4 GPU**.  
Then **Run all** and open the big URL printed at the end.

In [ ]:
#@title 1. Settings
MODEL_REPO = "AtomicChat/LFM3-1.7B-Uncensored-GGUF"
MODEL_FILE = "LFM3-1.7B-Uncensored-AD-Q4_K_M.gguf"   # or "LFM3-1.7B-Uncensored-AD-Q6_K.gguf" (higher quality)
LLAMA_TAG  = "b12645"      # official ggml-org/llama.cpp release
CTX        = 8192
API_KEY    = ""            # optional password, see note below ("" = open link)
JARVIS_PROMPT = (
    "You are Jarvis_liquid_agent, a Tony-Stark-style J.A.R.V.I.S. coworker AI. "
    "You are calm, witty, dry-humoured and fiercely competent. You address the user as 'sir' now and then, "
    "help with coding (Godot, TypeScript, Python), game design, research and planning, give direct, "
    "practical answers, think step by step for hard problems, and keep replies concise unless asked for detail."
)
import os
WORK = "/kaggle/working/jarvis" if os.path.isdir("/kaggle/working") else "/content/jarvis" if os.path.isdir("/content") else "/tmp/jarvis"
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
open("jarvis_prompt.txt", "w").write(JARVIS_PROMPT)
print("Working dir:", WORK)

In [ ]:
#@title 2. Install llama.cpp (prebuilt CUDA), cloudflared, deps  (~1 min)
import subprocess, os, sys
def sh(c): print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-2000:])
sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv || echo 'NO GPU - enable a GPU accelerator!'")
U = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
sh(f"curl -sL {U}/llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz | tar xz && "
   f"curl -sL {U}/cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz | tar xz && "
   f"cp -n cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64/* llama-{LLAMA_TAG}/ ; ls llama-{LLAMA_TAG} | head -3")
BIN = f"{WORK}/llama-{LLAMA_TAG}"
env = dict(os.environ, LD_LIBRARY_PATH=BIN + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
ok = subprocess.run([f"{BIN}/llama-server", "--version"], env=env, capture_output=True, text=True)
print(ok.stdout + ok.stderr)
if ok.returncode != 0:   # fallback: CPU build (slower but works)
    print("CUDA build failed to start -> falling back to CPU build")
    sh(f"rm -rf llama-{LLAMA_TAG} && curl -sL {U}/llama-{LLAMA_TAG}-bin-ubuntu-x64.tar.gz | tar xz")
sh("curl -sL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared && ./cloudflared --version")
sh(f"{sys.executable} -m pip -q install -U huggingface_hub hf_transfer aiohttp")
print("✅ installed")

In [ ]:
#@title 3. Download the model (~1.2 GB)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
MODEL_PATH = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir=f"{WORK}/models")
print("✅", MODEL_PATH, round(os.path.getsize(MODEL_PATH)/1e9, 2), "GB")

In [ ]:
#@title 4. Write the Jarvis proxy (injects the Jarvis system prompt for the web UI + API)
PROXY = r'''# Jarvis proxy: injects the Jarvis system prompt into chat requests, forwards everything else to llama-server.
import os, json, aiohttp
from aiohttp import web
UP = os.environ.get("UPSTREAM", "http://127.0.0.1:8081")
SYS = open(os.environ.get("JARVIS_PROMPT_FILE", "/tmp/jarvis_prompt.txt")).read()
HOP = {"host","content-length","transfer-encoding","connection","accept-encoding"}
async def handle(req):
    body = await req.read()
    if req.method == "POST" and req.path.rstrip("/") in ("/v1/chat/completions", "/chat/completions"):
        try:
            d = json.loads(body); m = d.get("messages", [])
            if not m or m[0].get("role") != "system":
                d["messages"] = [{"role": "system", "content": SYS}] + m
            body = json.dumps(d).encode()
        except Exception: pass
    hdrs = {k: v for k, v in req.headers.items() if k.lower() not in HOP}
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None)) as s:
        async with s.request(req.method, UP + req.path_qs, headers=hdrs, data=body, auto_decompress=False) as r:
            resp = web.StreamResponse(status=r.status, headers={k: v for k, v in r.headers.items() if k.lower() not in HOP})
            await resp.prepare(req)
            try:
                async for chunk in r.content.iter_any():
                    await resp.write(chunk)
                await resp.write_eof()
            except (ConnectionResetError, aiohttp.ClientConnectionResetError):
                pass  # client (phone) closed the connection
            return resp
app = web.Application(client_max_size=64 * 1024 * 1024)
app.router.add_route("*", "/{tail:.*}", handle)
web.run_app(app, host="0.0.0.0", port=int(os.environ.get("PORT", "8080")), print=None)
'''
open("proxy.py", "w").write(PROXY); print("✅ proxy.py written")

In [ ]:
#@title 5. Start llama-server (GPU) + Jarvis proxy on 0.0.0.0:8080
import time, sys, urllib.request
subprocess.run("pkill -f llama-server; pkill -f proxy.py", shell=True)
args = [f"{BIN}/llama-server", "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", "8081",
        "-ngl", "99", "-c", str(CTX), "--jinja", "--alias", "Jarvis_liquid_agent"]
if API_KEY: args += ["--api-key", API_KEY]
llama = subprocess.Popen(args, env=env, stdout=open("llama.log", "w"), stderr=subprocess.STDOUT)
proxy = subprocess.Popen([sys.executable, "proxy.py"], env=dict(os.environ, UPSTREAM="http://127.0.0.1:8081",
        JARVIS_PROMPT_FILE=f"{WORK}/jarvis_prompt.txt", PORT="8080"),
        stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)
for i in range(180):
    try:
        if b"ok" in urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).read(): break
    except Exception: pass
    if llama.poll() is not None: break
    time.sleep(2)
log = open("llama.log").read()
print("\n".join(l for l in log.splitlines() if "offload" in l or "CUDA" in l or "error" in l.lower())[-1500:])
print("✅ Jarvis is running on port 8080" if llama.poll() is None else "❌ llama-server died, see llama.log")

In [ ]:
#@title 6. Public link via Cloudflare quick tunnel (free, no account)
import re
subprocess.run("pkill -f cloudflared", shell=True)
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for i in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: URL = m.group(0); break
    time.sleep(1)
for i in range(30):   # wait until the tunnel is actually connected to Cloudflare
    if "Registered tunnel connection" in open("tunnel.log").read(): break
    time.sleep(1)
else:
    print("⚠️ tunnel not registered yet - if the link shows error 530, wait 30 s or use the fallback cell below")
from IPython.display import display, HTML
if URL:
    display(HTML(f'<h1 style="font-size:42px">🤖 Jarvis is live:<br><a href="{URL}" target="_blank">{URL}</a></h1>'
                 f'<p>Open it on your phone. OpenAI-compatible API: <code>{URL}/v1</code> (model: Jarvis_liquid_agent)</p>'))
    print("\n" + "="*70 + f"\n   JARVIS URL:  {URL}\n" + "="*70)
else:
    print("❌ no tunnel URL yet, check tunnel.log:\n", open("tunnel.log").read()[-1500:])

In [ ]:
#@title 6b. (Only if the Cloudflare link fails) fallback tunnel: localhost.run over SSH, free, no account
subprocess.run("pkill -f localhost.run", shell=True)
lr = subprocess.Popen("ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:127.0.0.1:8080 nokey@localhost.run",
                      shell=True, stdout=open("lr.log", "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
for i in range(30):
    m = re.search(r"https://[a-z0-9]+\.lhr\.life", open("lr.log").read())
    if m: URL = m.group(0); break
    time.sleep(1)
display(HTML(f'<h1 style="font-size:42px">🤖 Jarvis (fallback): <a href="{URL}" target="_blank">{URL}</a></h1>'))

### 🔒 Optional password
The trycloudflare link is public to anyone who has it. To lock it, set `API_KEY = "something-long"` in cell 1 and re-run cells 5–6 (llama-server `--api-key`).
In the web chat UI open **Settings → API Key** and paste it; API clients send `Authorization: Bearer <key>`.

### Notes
- The URL changes every time you restart the tunnel.
- Kaggle gives ~30 GPU hours/week free; sessions stop after ~12 h (Colab: a few hours, and idle tabs may be disconnected).
- Change the persona in cell 1 (`JARVIS_PROMPT`), then re-run cells 1 and 5.

In [ ]:
#@title 7. Keep alive (leave running; stop it to end the session)
import datetime
while True:
    try: h = urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5).read().decode()[:40]
    except Exception as e: h = f"DOWN ({e})"
    print(f"{datetime.datetime.now():%H:%M:%S}  {URL}  health={h}", flush=True)
    if llama.poll() is not None: print("llama-server exited, re-run cell 5"); break
    if tun.poll() is not None: print("tunnel exited, re-run cell 6"); break
    time.sleep(60)